In [ ]:
# %% [markdown]
# # DeepFM CTR 数据探索

# %%
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# 设置显示
pd.set_option('display.max_columns', 100)

# ============================================================
# 1. 添加项目路径
# ============================================================
project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"✅ 已添加项目路径: {project_root}")

# ============================================================
# 2. 导入模块
# ============================================================
from deepfm_ctr.DataProcess import *
from deepfm_ctr.config import DATA_CONFIG

print("✅ 导入成功")

# ============================================================
# 3. 加载数据
# ============================================================
processor = DataProcessor(
    data_dir=DATA_CONFIG['data_dir'],
    start_date=DATA_CONFIG['start_date'],
    split_date=DATA_CONFIG['split_date'],
    end_date=DATA_CONFIG['end_date'],
    val_ratio=DATA_CONFIG['val_ratio'],
    seed=DATA_CONFIG['seed']
)

raw_df = processor.load_data(sample_rate=0.01)
print(f"✅ 数据加载成功: {len(raw_df):,} 行, {len(raw_df.columns)} 列")

# 一次读取后完成时间切分；词汇表和归一化参数只在训练集上拟合
train_raw, val_raw, test_raw = processor.split_data(raw_df)
processor.fit(train_raw)
train = processor.transform(train_raw)
val = processor.transform(val_raw)
test = processor.transform(test_raw)

# 原始字段探索保留来源列，同时补齐声明式派生字段
df = processor._derive_features(raw_df)
print(f"训练集={len(train):,}, 验证集={len(val):,}, 测试集={len(test):,}")
# is_double_column 的派生规则由 DataProcess.py 统一执行

# ============================================================
# 5. 验证 is_double_column 是否存在
# ============================================================
print("\n" + "="*60)
print("验证")
print("="*60)
print(f"'is_double_column' in df.columns: {'is_double_column' in df.columns}")

# ============================================================
# 6. 特征存在性检查（现在应该都能找到了）
# ============================================================
print("\n" + "="*60)
print("特征存在性检查")
print("="*60)

existing_cat = [f for f in CATEGORICAL_FEATURES if f in df.columns]
missing_cat = [f for f in CATEGORICAL_FEATURES if f not in df.columns]

existing_num = [f for f in NUMERIC_FEATURES if f in df.columns]
missing_num = [f for f in NUMERIC_FEATURES if f not in df.columns]

print(f"类别特征: {len(existing_cat)}/{len(CATEGORICAL_FEATURES)} 存在")
if missing_cat:
    print(f"   ⚠️ 缺失: {missing_cat}")
else:
    print("   ✅ 所有类别特征都存在！")

print(f"\n数值特征: {len(existing_num)}/{len(NUMERIC_FEATURES)} 存在")
if missing_num:
    print(f"   ⚠️ 缺失: {missing_num[:5]}...")
else:
    print("   ✅ 所有数值特征都存在！")

# ============================================================
# 7. 缺失值分析（现在不会报错了）
# ============================================================
print("\n" + "="*60)
print("缺失值分析")
print("="*60)

# 使用所有特征（现在 is_double_column 已经存在了）
missing = df[CATEGORICAL_FEATURES + NUMERIC_FEATURES].isnull()
missing_rate = missing.mean() * 100
missing_rate = missing_rate[missing_rate > 0].sort_values(ascending=False)

if len(missing_rate) > 0:
    print(f"⚠️ 有 {len(missing_rate)} 个特征存在缺失值")
    print("\n缺失率最高的10个特征:")
    print(missing_rate.head(10))
    
    # 可视化
    fig, ax = plt.subplots(figsize=(14, max(6, len(missing_rate) * 0.3)))
    missing_rate.plot(kind='barh', ax=ax)
    ax.set_xlabel('缺失率 (%)')
    ax.set_title(f'特征缺失率分布 (共{len(missing_rate)}个特征有缺失)')
    ax.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='5% 阈值')
    ax.axvline(x=20, color='orange', linestyle='--', alpha=0.5, label='20% 阈值')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("✅ 无缺失值")

# ============================================================
# 8. 显示数据基本信息
# ============================================================
print("\n" + "="*60)
print("数据基本信息")
print("="*60)

if 'part_date' in df.columns:
    print(f"日期范围: {df['part_date'].min()} ~ {df['part_date'].max()}")

if LABEL in df.columns:
    pos_rate = df[LABEL].mean()
    print(f"正样本率: {pos_rate:.4f} ({pos_rate*100:.2f}%)")

print(f"\n数据形状: {df.shape}")
print("\n前5行:")
df.head()